In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def main():
    print("="*60)
    print(" [실험 1] 가중치 분화: 기하학적 궤적 변위 및 쌍성계(Binary System) 분석")
    print("="*60)

    torch.manual_seed(42)
    d_model = 32
    X = torch.randn(10, d_model) # 10개 토큰 입력

    # 가중치 초기화
    W_Q = nn.Parameter(torch.randn(d_model, d_model) * 0.1)
    W_K = nn.Parameter(torch.randn(d_model, d_model) * 0.1)
    W_V = nn.Parameter(torch.randn(d_model, d_model) * 0.1)

    optimizer = optim.SGD([W_Q, W_K, W_V], lr=0.5)

    # 초기 상태 백업
    W_Q_init, W_K_init, W_V_init = W_Q.clone().detach(), W_K.clone().detach(), W_V.clone().detach()

    print(" [진행] 문맥적 상호작용 하에 50스텝 가상 역전파 진행 중...")
    for step in range(50):
        optimizer.zero_grad()

        # 어텐션 연산 구조 재현
        Q, K, V = X @ W_Q, X @ W_K, X @ W_V
        scores = (Q @ K.T) / (d_model ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V

        # 목적 함수: 특정 문맥 패턴이 고도로 응집되도록 유도
        loss = out.std(dim=0).sum() - attn.diagonal().sum() * 0.1
        loss.backward()
        optimizer.step()

    # 가중치가 '이동한 방향 벡터' 정의
    diff_Q = (W_Q - W_Q_init).view(-1)
    diff_K = (W_K - W_K_init).view(-1)
    diff_V = (W_V - W_V_init).view(-1)

    # 이동 방향 간의 기하학적 사잇각(Cos-Sim) 계산
    cos_QK = torch.cosine_similarity(diff_Q, diff_K, dim=0).item()
    cos_QV = torch.cosine_similarity(diff_Q, diff_V, dim=0).item()
    cos_KV = torch.cosine_similarity(diff_K, diff_V, dim=0).item()

    print("\n [분석] 가중치 진화 이동 경로의 코사인 사잇각 분석")
    print(f" ΔW_Q <-> ΔW_K (상호작용 쌍): {cos_QK:+.4f}")
    print(f" ΔW_Q <-> ΔW_V (관계 vs 의미): {cos_QV:+.4f}")
    print(f" ΔW_K <-> ΔW_V (상호 vs 의미): {cos_KV:+.4f}")

    print("\n [해석]")
    if cos_QK > max(cos_QV, cos_KV):
        print(" -> [증명 완료] Q와 K는 서로의 꼬리를 무는 상호 관계 맵을 만들기 위해")
        print("    같은 역학적 평면 안에서 움직였지만, V는 완전히 독립된 다른 차원으로 분화되었습니다.")
    else:
        print(" -> 분화 양상을 보려면 태스크 복잡도나 학습 스텝을 늘려야 합니다.")
    print("="*60)

if __name__ == "__main__":
    main()